In [ ]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, InvalidVideoId, NoTranscriptFound, VideoUnavailable
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
embedding = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')

In [51]:
llm = HuggingFaceEndpoint(
    repo_id= "deepseek-ai/DeepSeek-V4-Flash", task = 'text-generation'
)

model = ChatHuggingFace(llm=llm)

In [52]:
video_id = "LPZh9BOjkQs"
ytt_api = YouTubeTranscriptApi()


try:
    transcript_list = ytt_api.fetch(video_id, languages=['en'])
    
    transcript = " ".join(chunk.text for chunk in transcript_list)
    
except(TranscriptsDisabled, InvalidVideoId, NoTranscriptFound, VideoUnavailable):
    print("Transcript not available or invalid video ID.")
    transcript = None
    
except AttributeError as e:
    print("Transcript API method not found:", e)
    

In [53]:
# Text Splitting

splitter = RecursiveCharacterTextSplitter(chunk_size = 900, chunk_overlap = 250)

chunks = splitter.create_documents([transcript])



In [54]:
chunks[10]

Document(metadata={}, page_content="operations, and as it does so, the hope is that each list of numbers is enriched to encode whatever information might be needed to make an accurate prediction of what word follows in the passage. At the end, one final function is performed on the last vector in this sequence, which now has had a chance to be influenced by all the other context from the input text, as well as everything the model learned during training, to produce a prediction of the next word. Again, the model's prediction looks like a probability for every possible next word. Although researchers design the framework for how each of these steps work, it's important to understand that the specific behavior is an emergent phenomenon based on how those hundreds of billions of parameters are tuned during training. This makes it incredibly challenging to determine why the model makes the exact predictions that it does.")

In [55]:
# Embedding generation and storing in vector store

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding        
)

In [56]:
vector_store.index_to_docstore_id

{0: 'c944bdcb-7057-4651-aebe-9c03184aab20',
 1: '6d9ff6b8-7a0b-4cb1-9edb-2b0db73957a3',
 2: '6f1a8a56-4c60-4762-885c-d34c6fe7f11d',
 3: '9decb5ab-f73c-4f8c-a3d7-ad64a69023a1',
 4: '476caf92-6f33-42c6-aac7-f39ec5177515',
 5: 'fa01bb75-6251-4b11-bb2c-b83c842e475b',
 6: 'f329daa2-382a-4b34-8bfa-95768a7727fc',
 7: 'ec61e71a-8bbc-4730-9726-871004d1b697',
 8: 'b3795fdd-b82c-4b1b-aef4-303a932487bd',
 9: '2dc1ecee-229a-4955-9946-b224a37826af',
 10: '1c6a6aed-9deb-4d22-aa3b-57b4720f4c0b',
 11: '8a1b7a63-3366-49fb-a80c-10a19517ab9e',
 12: '1a79f31a-e03c-45c3-9a06-1f011c5a5f31'}

In [57]:
vector_store.get_by_ids(['c944bdcb-7057-4651-aebe-9c03184aab20'])

[Document(id='c944bdcb-7057-4651-aebe-9c03184aab20', metadata={}, page_content='[Submit subtitle corrections at criblate.com]')]